# Phase 2 — Pattern Extraction (color / stripe / spot)

Runs `src/pattern_extractor/` against Phase 1's output (`data/extracted_fish/`) to produce one feature row per image across three independent dimensions: coloring (patternize-derived k-means clustering), spots (blob shape), stripes (region elongation + FFT periodicity).

**No GPU needed.** This is pure NumPy/SciPy/Pillow, deterministic and fast - CLAUDE.md notes this stage deliberately skips resumable per-image state for exactly that reason, a re-run just recomputes everything, cheaply. Use a **CPU runtime** here (`Runtime -> Change runtime type -> CPU`) to save your GPU quota for Phase 1.

**Prerequisite:** Phase 1 must already have produced output in `data/extracted_fish/` - run `Phase1_Fish_Extraction.ipynb` first.

See [README.md](../README.md) (Planned Approach, step 2) for the full citation and design reasoning (the *patternize* reimplementation, the three-dimension split's developmental-biology basis).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Open the same Drive-resident project Phase 1 used

This must resolve to the **same** `PROJECT_DIR` Phase 1 wrote `data/extracted_fish/` into - Phase 2 reads that output directly via the same relative-path layout. If this is a fresh Colab runtime that never ran Phase 1, this cell clones fresh onto Drive the same way Phase 1's notebook does; if Phase 1 already set it up, this just pulls any code updates.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/guptrishi01/Surgeonfish_Neural_Network_Phylogenetics.git"
PROJECT_DIR = Path("/content/drive/MyDrive/Surgeonfish_Neural_Network_Phylogenetics")

if not PROJECT_DIR.exists():
    print(f"Cloning into {PROJECT_DIR} (first time - pulls ~1.8GB, be patient)...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"{PROJECT_DIR} already exists - pulling latest code only.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)

In [ ]:
%cd {PROJECT_DIR}

## 3. Add the local package to the path

Base dependencies only (numpy, scipy, Pillow) - already in Colab's default image, no GPU extras needed for this phase.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))

import numpy, scipy, PIL
print("Base deps OK - numpy", numpy.__version__, "scipy", scipy.__version__, "Pillow", PIL.__version__)

## 4. Confirm Phase 1's output is actually there - and check for fully-dropped species

`fish_extractor` only creates a species folder in `data/extracted_fish/` once at least one of that species' images was *accepted* - a species where every image ended up excluded or still flagged has no folder at all, not an empty one, and Phase 2 would silently produce zero rows for it rather than erroring. With 604/1,460 images excluded in the real run, this is worth actually checking rather than assuming it didn't happen - compares the full 64-species list from `data/raw_images/` (untouched by Phase 1, so always complete) against what `data/extracted_fish/` actually has.

In [ ]:
extracted_root = PROJECT_DIR / "data" / "extracted_fish"
raw_root = PROJECT_DIR / "data" / "raw_images"

raw_species = {
    f"{genus.name}/{species.name}"
    for genus in raw_root.iterdir() if genus.is_dir()
    for species in genus.iterdir() if species.is_dir()
}
extracted_species = {
    f"{genus.name}/{species.name}"
    for genus in extracted_root.iterdir() if genus.is_dir()
    for species in genus.iterdir() if species.is_dir()
} if extracted_root.exists() else set()

print(f"{len(raw_species)} species in data/raw_images/, {len(extracted_species)} have at least "
      f"one accepted image in data/extracted_fish/.")

missing = sorted(raw_species - extracted_species)
if missing:
    print(f"\n{len(missing)} species have ZERO accepted images - every image was excluded/flagged:")
    for m in missing:
        print(f"  {m}")
    print("\nThese won't appear in pattern_features.csv at all. Worth deciding now whether that's "
          "expected (e.g. a species with very few source images to begin with) or worth revisiting "
          "in Phase 1 before continuing.")
elif not extracted_species:
    print("Nothing here yet - run Phase1_Fish_Extraction.ipynb first.")
else:
    print("All species have at least one accepted image.")

## 5. Run pattern extraction

Writes one feature row per image (color/spot/stripe columns, plus an `is_reference` column marking the curated seed photo vs. GBIF field images - see README.md's Planned Approach step 3 for why that distinction matters for the aggregation step that comes next in Phase 3) to `reports/pattern_features.csv`.

Logging is configured first so you get a `[n/64] <species>: <count> processed` line per species as it runs - this stage has no resumable state (a re-run just recomputes everything, see the package docstring), so this is the only progress visibility available while it's running, not something you can check after the fact. Colour clustering's k-means fit is the expensive part per image; it now runs on at most 50,000 randomly-sampled masked-in pixels rather than every single one (real, non-resized crops can have millions), which should keep this to well under an hour total - if a single species' line takes more than a minute or two to appear, that's worth flagging back rather than assuming it'll finish.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s")

from pattern_extractor.config import PipelineConfig
from pattern_extractor.pipeline import PatternExtractorPipeline

config = PipelineConfig()  # relative paths, same PROJECT_DIR layout Phase 1 used
rows = PatternExtractorPipeline(config).run()
print(f"Wrote {len(rows)} feature row(s) to {config.output_csv_path.resolve()}")

## 6. Quick sanity check, including real per-species image counts

The README's Phase 3 plan already documents specific sparse-species concerns (e.g. *Naso maculatus* down to a single image) based on Phase 0/1's *pre-review* numbers - this recomputes it from what Phase 1's real review process actually left behind, since exclusions during review could have thinned any species further, not just the ones already flagged as sparse.

In [ ]:
import csv
from collections import Counter

with open(config.output_csv_path, newline="", encoding="utf-8") as f:
    reader = list(csv.DictReader(f))

n_species_seen = len({row["image_key"].split("/")[1] for row in reader})
n_reference = sum(1 for row in reader if row["is_reference"] == "True")
print(f"{len(reader)} image row(s) across {n_species_seen} species; {n_reference} marked is_reference.")

per_species = Counter(row["image_key"].split("/")[1] for row in reader)
sparse = sorted((n, sp) for sp, n in per_species.items() if n <= 5)
print(f"\n{len(sparse)} species with 5 or fewer images after Phase 1's real review:")
for n, sp in sparse:
    print(f"  {sp}: {n}")

reader[:3]

## Next: Phase 3

`reports/pattern_features.csv` is Phase 3's input (per-species aggregation + distance matrices - see README.md's Planned Approach, step 3). It's already saved under Drive; pull it down locally, or keep working from Drive, to continue there.